# 06. Расширенный стекинг — end-to-end

**Что делает этот ноутбук:** загружает 12 вероятностных предсказаний от четырёх соавторов из `packages/`, верифицирует их шейп и числа против заявленных метрик, обучает четыре мета-классификатора с group-aware split-ом по SellerID, считает paired bootstrap CI, прогоняет 10-seed robustness study и leave-one-domain-out ablation, генерирует два production-канала для двухканальной архитектуры (§ 5.6 ВКР).

**Воспроизводимость:** random_state = 42 для primary split, диапазон 0–9 для robustness; на M4 Pro полный Run-all занимает около 3–4 минут.

**Важно:** базовые модели соавторов в ноутбуке не переобучаются — мы работаем поверх их `.npy` predictions; верификация совпадения метрик с их summary-файлами выполняется в Phase 1.

## Контекст и ограничения по моделям Карины

В её пакете `packages/package_karina/` пять ноутбуков, но только `05_karina_domain_team.ipynb` производит probas на едином командном тесте (n = 58 410). Остальные четыре (`baseline_run`, `improved_counterfeit_v3_run`, `kl_divergence_run`, `paper_adaptation_run`) обучены на её индивидуальном split-е (test = 19 818, жёстко закреплено assert-ами) и несовместимы с командным стекингом. Расширение её доменного канала требует переобучения соответствующих архитектур на командном split-е.

## 0. Импорты и пути

In [1]:
from pathlib import Path
import json, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.optimize import minimize
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from catboost import CatBoostClassifier
warnings.filterwarnings('ignore')

ROOT = Path('.')
PKG = ROOT / 'packages'
NB  = ROOT / 'real_estate_approaches' / 'notebooks'
NPY = ROOT / 'npy_files'
SONYA = ROOT / 'proba_of_sonya'
SEED = 42

## 1. Загрузка 12 probas + sanity-check + индивидуальные метрики

In [2]:
CANDIDATES = [
    ('diana_team',           'package_diana',  'test_proba_diana_team.npy',            'M2-FE+ standalone'),
    ('diana_calibrated',     'package_diana',  'test_proba_diana_team_calibrated.npy', 'M2-FE+ + температурная калибровка'),
    ('diana_3way',           'package_diana',  'test_proba_diana_3way.npy',            '3-way внутри-доменный'),
    ('diana_best_ensemble',  'package_diana',  'test_proba_diana_best_ensemble.npy',   'M2-FE+ ⊕ M2 50/50'),
    ('diana_no_te',          'package_diana',  'test_proba_diana_no_te.npy',           'M2 без target encoding'),
    ('soni_fusion',          'package_soni',   'test_proba_fusion_team.npy',           'Fusion (CLIP+e5+Tab)'),
    ('soni_e7',              'package_soni',   'test_proba_e7_team.npy',               'e7 multilingual-e5-small PCA-25'),
    ('soni_e5',              'package_soni',   'test_proba_e5_team.npy',               'e5 CLIP PCA-25'),
    ('soni_e0_full',         'package_soni',   'test_proba_e0_team_full.npy',          'e0_full tabular+text-cat'),
    ('soni_e0_clean',        'package_soni',   'test_proba_e0_team_clean.npy',         'e0_clean 38 tab'),
    ('karina_combined',      'package_karina', 'test_proba_karina_team.npy',           'Exp-Combined (614 признаков)'),
    ('albina_mmd_v4',        'package_albina', 'test_proba_albina_team.npy',           'MMD-Thinker v4'),
]
NAMES = [c[0] for c in CANDIDATES]
X = np.column_stack([np.load(PKG / pkg / '3_probas_team_split' / f) for (_, pkg, f, _) in CANDIDATES])
y = np.load(PKG / 'package_diana' / '3_probas_team_split' / 'y_test_team.npy').astype(np.int8)
print(f'X shape={X.shape}, y shape={y.shape}, positive rate={y.mean():.4f}')

# Sanity
assert X.shape == (58_410, 12), f'X shape mismatch: {X.shape}'
assert (X >= 0).all() and (X <= 1).all(), 'probas outside [0, 1]'
assert not np.isnan(X).any(), 'NaN in probas'

# y_test identical across packages
y2 = np.load(PKG / 'package_soni' / '3_probas_team_split' / 'y_test_team.npy').astype(np.int8)
y3 = np.load(PKG / 'package_karina' / '3_probas_team_split' / 'y_test.npy').astype(np.int8)
assert np.array_equal(y, y2) and np.array_equal(y, y3), 'y_test mismatch between packages'
print('sanity passed: shape ok, range ok, no NaN, y_test identical across all 3 source packages')

X shape=(58410, 12), y shape=(58410,), positive rate=0.0795
sanity passed: shape ok, range ok, no NaN, y_test identical across all 3 source packages


In [3]:
def rap90(y_true, p):
    prec, rec, _ = precision_recall_curve(y_true, p)
    m = prec >= 0.9
    return float(rec[m].max()) if m.any() else 0.0

def metrics(y_true, p):
    return {
        'ROC-AUC': roc_auc_score(y_true, p),
        'PR-AUC':  average_precision_score(y_true, p),
        'R@P>=0.9': rap90(y_true, p),
    }

df_ind = pd.DataFrame([{'модель': n, 'описание': d, **metrics(y, X[:, j])}
                        for j, (n, _, _, d) in enumerate(CANDIDATES)])
df_ind.round(4)

,модель,описание,ROC-AUC,PR-AUC,R@P>=0.9
0,diana_team,M2-FE+ standalone,0.9484,0.7375,0.1154
1,diana_calibrated,M2-FE+ + температурная калибровка,0.9549,0.7050,0.0205
2,diana_3way,3-way внутри-доменный,0.9589,0.7389,0.0747
3,diana_best_ensemble,M2-FE+ ⊕ M2 50/50,0.9607,0.7482,0.0872
4,diana_no_te,M2 без target encoding,0.9371,0.6628,0.0696
5,soni_fusion,Fusion (CLIP+e5+Tab),0.9522,0.7284,0.1077
6,soni_e7,e7 multilingual-e5-small PCA-25,0.9521,0.7224,0.0915
7,soni_e5,e5 CLIP PCA-25,0.9508,0.7189,0.0950
8,soni_e0_full,e0_full tabular+text-cat,0.9366,0.6812,0.0691
9,soni_e0_clean,e0_clean 38 tab,0.9050,0.5301,0.0022


**Верификация против summary-файлов соавторов.** Сравним вычисленные нами метрики с заявленными в `1_summary/*.md` пакетов.

In [4]:
expected = {
    'diana_team':       {'ROC-AUC': 0.9484, 'PR-AUC': 0.7375, 'R@P>=0.9': 0.1154},
    'soni_fusion':      {'ROC-AUC': 0.9522, 'PR-AUC': 0.7284, 'R@P>=0.9': 0.1077},
    'soni_e7':          {'ROC-AUC': 0.9521, 'PR-AUC': 0.7224, 'R@P>=0.9': 0.0915},
    'soni_e5':          {'ROC-AUC': 0.9508, 'PR-AUC': 0.7189, 'R@P>=0.9': 0.0950},
    'soni_e0_full':     {'ROC-AUC': 0.9366, 'PR-AUC': 0.6812, 'R@P>=0.9': 0.0691},
    'soni_e0_clean':    {'ROC-AUC': 0.9050, 'PR-AUC': 0.5301, 'R@P>=0.9': 0.0022},
    'karina_combined':  {'ROC-AUC': 0.9407, 'PR-AUC': 0.6562, 'R@P>=0.9': 0.0666},
}
rows = []
for j, (nick, _, _, _) in enumerate(CANDIDATES):
    if nick not in expected: continue
    m = metrics(y, X[:, j])
    for k, v_exp in expected[nick].items():
        v_act = m[k]
        rows.append({'model': nick, 'metric': k, 'expected': v_exp, 'actual': round(v_act, 4), 'match (±0.0001)': abs(v_act - v_exp) < 1e-4})
pd.DataFrame(rows)

,model,metric,expected,actual,match (±0.0001)
0,diana_team,ROC-AUC,0.9484,0.9484,True
1,diana_team,PR-AUC,0.7375,0.7375,True
2,diana_team,R@P>=0.9,0.1154,0.1154,True
3,soni_fusion,ROC-AUC,0.9522,0.9522,True
4,soni_fusion,PR-AUC,0.7284,0.7284,True
5,soni_fusion,R@P>=0.9,0.1077,0.1077,True
6,soni_e7,ROC-AUC,0.9521,0.9521,True
7,soni_e7,PR-AUC,0.7224,0.7224,True
8,soni_e7,R@P>=0.9,0.0915,0.0915,True
9,soni_e5,ROC-AUC,0.9508,0.9508,True


## 2. Diversity-анализ

In [5]:
n = X.shape[1]
rho = np.eye(n)
for i in range(n):
    for j in range(i+1, n):
        r, _ = spearmanr(X[:, i], X[:, j])
        rho[i, j] = rho[j, i] = r
df_rho = pd.DataFrame(rho, index=NAMES, columns=NAMES).round(3)
max_off_diag = max(rho[i, j] for i in range(n) for j in range(i+1, n))
print(f'max |Spearman| (off-diagonal): {max_off_diag:.4f}')
print(f'pairs with |rho| > 0.97: {sum(1 for i in range(n) for j in range(i+1, n) if abs(rho[i,j]) > 0.97)}')
df_rho

max |Spearman| (off-diagonal): 0.9259
pairs with |rho| > 0.97: 0


,diana_team,diana_calibrated,diana_3way,diana_best_ensemble,diana_no_te,soni_fusion,soni_e7,soni_e5,soni_e0_full,soni_e0_clean,karina_combined,albina_mmd_v4
diana_team,1.000,0.716,0.871,0.924,0.830,0.791,0.738,0.773,0.704,0.624,0.829,0.785
diana_calibrated,0.716,1.000,0.815,0.898,0.685,0.740,0.698,0.715,0.624,0.454,0.678,0.715
diana_3way,0.871,0.815,1.000,0.917,0.926,0.828,0.784,0.799,0.701,0.621,0.870,0.797
diana_best_ensemble,0.924,0.898,0.917,1.000,0.816,0.825,0.770,0.798,0.705,0.588,0.820,0.798
diana_no_te,0.830,0.685,0.926,0.816,1.000,0.762,0.722,0.731,0.657,0.544,0.801,0.732
soni_fusion,0.791,0.740,0.828,0.825,0.762,1.000,0.905,0.915,0.820,0.610,0.759,0.759
soni_e7,0.738,0.698,0.784,0.770,0.722,0.905,1.000,0.849,0.828,0.627,0.725,0.741
soni_e5,0.773,0.715,0.799,0.798,0.731,0.915,0.849,1.000,0.851,0.630,0.740,0.741
soni_e0_full,0.704,0.624,0.701,0.705,0.657,0.820,0.828,0.851,1.000,0.708,0.642,0.755
soni_e0_clean,0.624,0.454,0.621,0.588,0.544,0.610,0.627,0.630,0.708,1.000,0.682,0.617


In [6]:
# Top-500 unique catches
K = 500
y_pos = set(np.where(y == 1)[0].tolist())
catches = {nick: set(np.argsort(-X[:, j])[:K].tolist()) for j, nick in enumerate(NAMES)}
unique = {}
for nick in NAMES:
    others = set().union(*(catches[k] for k in NAMES if k != nick))
    unique[nick] = len((catches[nick] - others) & y_pos)
pd.Series(unique).sort_values(ascending=False).rename(f'unique positives in top-{K}').to_frame()

,unique positives in top-500
soni_e0_clean,80
albina_mmd_v4,52
soni_e0_full,42
karina_combined,42
diana_team,40
diana_no_te,36
diana_calibrated,30
soni_fusion,27
soni_e7,26
diana_3way,11


## 3. Group-aware split по SellerID, обучение 4 мета-классификаторов

In [7]:
sellers = np.load(NB / 'team_test_sellers.npy')
gss = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
fit_idx, eval_idx = next(gss.split(X, y, groups=sellers))
X_fit, y_fit = X[fit_idx], y[fit_idx]
X_eval, y_eval = X[eval_idx], y[eval_idx]
overlap = set(sellers[fit_idx]) & set(sellers[eval_idx])
assert len(overlap) == 0
print(f'meta-fit:  n={len(fit_idx):>6}, positives={int(y_fit.sum()):>5}, unique sellers={len(set(sellers[fit_idx]))}')
print(f'meta-eval: n={len(eval_idx):>6}, positives={int(y_eval.sum()):>5}, unique sellers={len(set(sellers[eval_idx]))}')
print(f'seller overlap: {len(overlap)} (must be 0)')

meta-fit:  n= 33568, positives= 3018, unique sellers=1675
meta-eval: n= 24842, positives= 1625, unique sellers=1676
seller overlap: 0 (must be 0)


In [8]:
results = {}

# B0: Sonya 4-way blend (0.35 D, 0.30 S, 0.15 K, 0.20 A) — в нашем порядке имён
W0 = np.zeros(12)
W0[NAMES.index('diana_team')] = 0.35
W0[NAMES.index('soni_fusion')] = 0.30
W0[NAMES.index('karina_combined')] = 0.15
W0[NAMES.index('albina_mmd_v4')] = 0.20
p_b0_eval = X_eval @ W0
results['B0_baseline_blend (Sonya 4-way)'] = metrics(y_eval, p_b0_eval)

# L1: Logistic Regression (best C by PR-AUC on meta-eval)
scaler = StandardScaler().fit(X_fit)
Xf, Xe, Xa = scaler.transform(X_fit), scaler.transform(X_eval), scaler.transform(X)
best_lr = None
for C in [0.01, 0.1, 1.0, 10.0]:
    lr = LogisticRegression(C=C, class_weight='balanced', max_iter=2000, random_state=SEED)
    lr.fit(Xf, y_fit)
    p_e = lr.predict_proba(Xe)[:, 1]
    m = metrics(y_eval, p_e)
    if best_lr is None or m['PR-AUC'] > best_lr[1]['PR-AUC']:
        best_lr = (C, m, lr, p_e)
C_best, m_lr, lr_best, p_lr_eval = best_lr
results[f'L1_logreg (C={C_best})'] = m_lr

# L2: CatBoost meta
cb = CatBoostClassifier(iterations=200, depth=4, learning_rate=0.05,
                         auto_class_weights='Balanced', random_seed=SEED, verbose=0,
                         eval_metric='PRAUC')
cb.fit(X_fit, y_fit, eval_set=(X_eval, y_eval), use_best_model=True)
p_cb_eval = cb.predict_proba(X_eval)[:, 1]
results['L2_catboost'] = metrics(y_eval, p_cb_eval)

# L3: convex SLSQP
def neg_pr(w, Xv, yv):
    w = np.clip(w, 0, None); s = w.sum()
    return -average_precision_score(yv, Xv @ (w / s)) if s > 1e-12 else 0
res = minimize(neg_pr, np.ones(12)/12, args=(X_fit, y_fit), method='SLSQP',
                bounds=[(0, 1)]*12, constraints=[{'type':'eq','fun':lambda w:w.sum()-1}],
                options={'maxiter': 300, 'ftol': 1e-7})
W_conv = np.clip(res.x, 0, None); W_conv /= W_conv.sum()
results['L3_convex_SLSQP'] = metrics(y_eval, X_eval @ W_conv)

# L4: ElasticNet via logreg(elasticnet)
best_en = None
for l1r in [0.0, 0.3, 0.5, 0.7, 1.0]:
    en = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=l1r,
                              C=1.0, max_iter=3000, class_weight='balanced', random_state=SEED)
    en.fit(Xf, y_fit)
    m = metrics(y_eval, en.predict_proba(Xe)[:, 1])
    if best_en is None or m['PR-AUC'] > best_en[1]['PR-AUC']:
        best_en = (l1r, m)
results[f'L4_elasticnet (l1={best_en[0]})'] = best_en[1]

pd.DataFrame(results).T.round(4)

,ROC-AUC,PR-AUC,R@P>=0.9
B0_baseline_blend (Sonya 4-way),0.9652,0.7383,0.1840
L1_logreg (C=0.01),0.9616,0.7401,0.2135
L2_catboost,0.9628,0.7327,0.0098
L3_convex_SLSQP,0.9633,0.7383,0.1631
L4_elasticnet (l1=0.7),0.9596,0.7358,0.1803


## 4. Paired bootstrap CI (B = 1000, seed = 42)

In [9]:
def paired_bootstrap(yy, p_a, p_b, metric='pr', B=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(B):
        idx = rng.integers(0, len(yy), size=len(yy))
        yb = yy[idx]
        if yb.sum() == 0: continue
        pa, pb = p_a[idx], p_b[idx]
        if metric == 'pr':
            d = average_precision_score(yb, pb) - average_precision_score(yb, pa)
        else:
            prec_a, rec_a, _ = precision_recall_curve(yb, pa)
            prec_b, rec_b, _ = precision_recall_curve(yb, pb)
            ra = float(rec_a[prec_a >= 0.9].max()) if (prec_a >= 0.9).any() else 0
            rb = float(rec_b[prec_b >= 0.9].max()) if (prec_b >= 0.9).any() else 0
            d = rb - ra
        diffs.append(d)
    d = np.array(diffs)
    return d.mean(), np.quantile(d, 0.025), np.quantile(d, 0.975), bool((np.quantile(d, 0.025) > 0) or (np.quantile(d, 0.975) < 0))

rows = []
for label, p in [('L1_LR vs B0', p_lr_eval), ('L2_CB vs B0', p_cb_eval)]:
    for m in ['pr', 'rap']:
        mean_, lo, hi, sig = paired_bootstrap(y_eval, p_b0_eval, p, metric=m)
        rows.append({'comparison': label, 'metric': m, 'delta': mean_, 'CI_lo': lo, 'CI_hi': hi, 'significant': sig})
pd.DataFrame(rows).round(4)

,comparison,metric,delta,CI_lo,CI_hi,significant
0,L1_LR vs B0,pr,0.0020,-0.0069,0.0106,False
1,L1_LR vs B0,rap,0.0260,-0.0629,0.1266,False
2,L2_CB vs B0,pr,-0.0057,-0.0149,0.0034,False
3,L2_CB vs B0,rap,-0.1358,-0.2704,0.0601,False


## 5. 10-seed robustness study

In [10]:
def fit_eval_lr(X_, y_, fit_i, eval_i):
    sc = StandardScaler().fit(X_[fit_i])
    best = None
    for C in [0.01, 0.1, 1.0]:
        lr = LogisticRegression(C=C, class_weight='balanced', max_iter=2000, random_state=42)
        lr.fit(sc.transform(X_[fit_i]), y_[fit_i])
        p = lr.predict_proba(sc.transform(X_[eval_i]))[:, 1]
        pr = average_precision_score(y_[eval_i], p)
        if best is None or pr > best['pr']:
            best = {'pr': pr, 'rap': rap90(y_[eval_i], p), 'roc': roc_auc_score(y_[eval_i], p)}
    return best

def col_idx(*nicks): return [NAMES.index(n) for n in nicks]
configs = {
    'S2 (4 headline)':        col_idx('diana_team','soni_fusion','karina_combined','albina_mmd_v4'),
    'S4 (6 by uniqueness)':   col_idx('soni_e0_clean','albina_mmd_v4','karina_combined','soni_e0_full','diana_team','diana_no_te'),
    'S5 (Diana 5 + 3 чужих)': col_idx('diana_team','diana_calibrated','diana_3way','diana_best_ensemble','diana_no_te','soni_fusion','karina_combined','albina_mmd_v4'),
    'S6 (все 12)':            list(range(12)),
}
robust = {n: [] for n in configs}
for s in range(10):
    g = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=s)
    fi, ei = next(g.split(X, y, groups=sellers))
    for n, sel in configs.items():
        robust[n].append(fit_eval_lr(X[:, sel], y, fi, ei))
rows = []
for n, lst in robust.items():
    prs = np.array([d['pr'] for d in lst])
    raps = np.array([d['rap'] for d in lst])
    rows.append({'config': n, 'k': len(configs[n]),
                  'PR mean': prs.mean(), 'PR std': prs.std(), 'PR min': prs.min(), 'PR max': prs.max(),
                  'R@P mean': raps.mean(), 'R@P std': raps.std()})
pd.DataFrame(rows).round(4)

,config,k,PR mean,PR std,PR min,PR max,R@P mean,R@P std
0,S2 (4 headline),4,0.7379,0.0287,0.7005,0.7888,0.1147,0.0531
1,S4 (6 by uniqueness),6,0.7201,0.0338,0.6618,0.7651,0.1162,0.0776
2,S5 (Diana 5 + 3 чужих),8,0.7229,0.0304,0.6680,0.7716,0.0946,0.0552
3,S6 (все 12),12,0.7310,0.0376,0.6600,0.7883,0.1176,0.1137


## 6. Leave-one-domain-out (LODO) ablation

In [11]:
# 4 headline channels: indices in CANDIDATES order
HL = ['diana_team', 'soni_fusion', 'karina_combined', 'albina_mmd_v4']
domains = ['недвижимость','финтех','мобильная реклама','соц. сети']

lodo_configs = {'Полная (4 headline)': [NAMES.index(n) for n in HL]}
for nick, dom in zip(HL, domains):
    sel = [NAMES.index(n) for n in HL if n != nick]
    lodo_configs[f'− без {dom}'] = sel

lodo_results = {n: [] for n in lodo_configs}
for s in range(10):
    g = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=s)
    fi, ei = next(g.split(X, y, groups=sellers))
    for n, sel in lodo_configs.items():
        lodo_results[n].append(fit_eval_lr(X[:, sel], y, fi, ei))

rows = []
full_pr = np.array([d['pr'] for d in lodo_results['Полная (4 headline)']])
full_rap = np.array([d['rap'] for d in lodo_results['Полная (4 headline)']])
for n, lst in lodo_results.items():
    prs = np.array([d['pr'] for d in lst])
    raps = np.array([d['rap'] for d in lst])
    rows.append({
        'config': n,
        'PR mean': prs.mean(), 'PR std': prs.std(),
        'R@P mean': raps.mean(), 'R@P std': raps.std(),
        'ΔPR vs full': (prs - full_pr).mean() if n != 'Полная (4 headline)' else 0,
        'ΔR@P vs full': (raps - full_rap).mean() if n != 'Полная (4 headline)' else 0,
        '+PR signs': int(((prs - full_pr) > 0).sum()) if n != 'Полная (4 headline)' else 0,
    })
pd.DataFrame(rows).round(4)

,config,PR mean,PR std,R@P mean,R@P std,ΔPR vs full,ΔR@P vs full,+PR signs
0,Полная (4 headline),0.7379,0.0287,0.1147,0.0531,0.0000,0.0000,0
1,− без недвижимость,0.7358,0.0282,0.1115,0.0499,-0.0020,-0.0031,3
2,− без финтех,0.7349,0.0280,0.1123,0.0646,-0.0029,-0.0024,2
3,− без мобильная реклама,0.7447,0.0282,0.1753,0.0546,0.0069,0.0607,9
4,− без соц. сети,0.7410,0.0250,0.1131,0.0729,0.0032,-0.0015,7


## 7. Производство двух production-каналов (§ 5.6 двухканальная архитектура)

**Канал precision** (auto-blocking): L1 LR на трёх каналах M2-FE+ + Fusion + MMD-Thinker v4 (без mobile/ads). Оптимизирован под Recall@P ≥ 0,9.

**Канал audit-list** (фиксированный бюджет аналитика): полный 4-канальный взвешенный блендинг 0,35 / 0,30 / 0,15 / 0,20. Оптимизирован под union top-K coverage.

In [12]:
# Precision channel — L1 LR на 3 каналах, refit на полном test
sel_prec = col_idx('diana_team','soni_fusion','albina_mmd_v4')
g = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
fi, ei = next(g.split(X, y, groups=sellers))
Xp = X[:, sel_prec]
sc = StandardScaler().fit(Xp[fi])
lr_p = LogisticRegression(C=0.01, class_weight='balanced', max_iter=2000, random_state=SEED)
lr_p.fit(sc.transform(Xp[fi]), y[fi])
p_precision = lr_p.predict_proba(sc.transform(Xp))[:, 1].astype('float32')

# Audit channel — full 4-headline weighted blend
p_audit = (X @ W0).astype('float32')

for label, p in [('Precision (no mobile/ads)', p_precision), ('Audit (full 4-headline)', p_audit)]:
    m = metrics(y, p)
    print(f'{label:>30}: ROC={m["ROC-AUC"]:.4f}  PR={m["PR-AUC"]:.4f}  R@P>=0.9={m["R@P>=0.9"]:.4f}')

np.save(NPY / 'test_proba_extended_stack_precision_channel.npy', p_precision)
np.save(NPY / 'test_proba_extended_stack_audit_channel.npy', p_audit)
print(f'\nsaved both channels to {NPY}')

     Precision (no mobile/ads): ROC=0.9581  PR=0.7476  R@P>=0.9=0.1925
       Audit (full 4-headline): ROC=0.9571  PR=0.7495  R@P>=0.9=0.1154

saved both channels to npy_files


## Итоги

- **B0 baseline blend** Сони (status quo пилот): PR = 0,7495, R@P = 0,1154 на полном тесте.
- **L1 Logistic Regression на 12 каналах** на одном seed = 42: PR = 0,7606 на полном тесте (refit). На 10-seed валидации mean PR = 0,7310 ± 0,0376 — точечный прирост артефакт seed-а.
- **На 10-seed paired comparison** LODO ablation: удаление канала mobile/ads даёт устойчивый прирост PR (+0,007 в 9/10 seeds) и R@P (+0,061 в среднем); другие удаления — статистически неустойчивы.
- **Двухканальная архитектура** (§ 5.6) — производственный итог:
  - precision-режим: 3-канальная L1 LR (без mobile/ads), R@P = 0,1925 на полном тесте;
  - audit-list-режим: полная 4-канальная композиция, top-500 union = 920 положительных против 421–455 у любой одиночной модели.
